<a href="https://colab.research.google.com/github/daanilm14/gpu-short-course/blob/main/compute_pi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1: Compute pi on the GPU

## Tasks

1. Write a CPU-program that computes pi using geometry and statistics
2. Port this program to the GPU
3. Optimize

## Solution

In [ ]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.0 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=0e8785371e7da7f81a436c75e37068c2aef53c0555366d452b6bc04344766e8e
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [ ]:
import numpy as np
import time
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

def compute_pi_cpu(n):
    # Generar puntos aleatorios x e y entre 0 y 1
    x = np.random.rand(n)
    y = np.random.rand(n)

    # Calcular distancia al origen (x^2 + y^2 <= 1)
    # Usamos la suma de booleanos para contar aciertos
    inside_circle = (x**2 + y**2) <= 1

    return 4 * np.sum(inside_circle) / n

<-- see `compute_pi_solution.ipynb`

In [ ]:
tic = time.time()
print(compute_pi_cpu(51200000))
toc = time.time()

print("Time to execute cpu version: {:f} seconds".format(toc - tic))

3.14152875
Time to execute cpu version: 1.261501 seconds


In [ ]:
pi_kernel_src = """
#include <math.h>

// Generador de números pseudoaleatorios (LCG)
__device__ float generateRandomNumber(long& last_draw) {
    last_draw = last_draw * 1103515245 + 12345;
    long abs = last_draw & 0x7fffffff;
    return abs / 2147483648.0f;
}

__global__ void computePi(unsigned int* inside, unsigned int seed) {
    // Memoria compartida para los hilos del bloque (512 hilos)
    __shared__ unsigned int inside_shared[512];

    unsigned int tid = threadIdx.x;
    unsigned int gid = blockIdx.x * blockDim.x + threadIdx.x;

    // Simulación del dardo
    long rand_seed = seed + gid;
    float x = generateRandomNumber(rand_seed);
    float y = generateRandomNumber(rand_seed);

    // Guardar 1 si está dentro del círculo, 0 si no
    inside_shared[tid] = (x*x + y*y <= 1.0f) ? 1 : 0;

    __syncthreads(); // Sincronizar para que todos escriban en shared memory

    // REDUCCIÓN PARALELA: Suma jerárquica dentro del bloque
    for (unsigned int s = blockDim.x / 2; s > 32; s >>= 1) {
        if (tid < s) {
            inside_shared[tid] += inside_shared[tid + s];
        }
        __syncthreads();
    }

    // Reducción final para el último Warp (32 hilos) sin sincronización explícita
    if (tid < 32) {
        volatile unsigned int* p = inside_shared;
        p[tid] += p[tid + 32];
        p[tid] += p[tid + 16];
        p[tid] += p[tid + 8];
        p[tid] += p[tid + 4];
        p[tid] += p[tid + 2];
        p[tid] += p[tid + 1];
    }

    // El hilo 0 de cada bloque guarda el subtotal en la memoria global
    if (tid == 0) {
        inside[blockIdx.x] = inside_shared[0];
    }
}
"""

# Compilar el código C++ en la GPU
mod = SourceModule(pi_kernel_src)
func = mod.get_function("computePi")

In [ ]:
def compute_pi_gpu_optimized(n_points, threads_per_block=512):
    # n_points debe ser múltiplo de 512 para que el grid sea exacto
    num_blocks = n_points // threads_per_block

    # Reservamos memoria para 1 resultado por bloque (mucho menos tráfico PCIe)
    inside_gpu = cuda.mem_alloc(num_blocks * 4)

    # Lanzar el kernel
    func(inside_gpu, np.uint32(time.time()),
         block=(threads_per_block, 1, 1),
         grid=(num_blocks, 1, 1))

    # Traer los subtotales a la CPU
    subtotales = np.empty(num_blocks, dtype=np.uint32)
    cuda.memcpy_dtoh(subtotales, inside_gpu)

    # Cálculo final: la CPU solo suma los subtotales de los bloques
    total_inside = np.sum(subtotales)
    return 4.0 * total_inside / n_points

In [ ]:
tic = time.time()
print(compute_pi_cpu(51200000))
toc = time.time()

print("Time to execute cpu version: {:f} seconds".format(toc - tic))

3.141376953125
Time to execute cpu version: 1.312177 seconds
